In [1]:
%pip install -r requirements-offline.txt --quiet

/Users/utarn/projects/datascience-metrology/.check-env/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# Solution — Module M2: จัดการตารางด้วย pandas

> **เฉลย — ผู้สอนจะเปิดเผยในห้องหลังจบช่วงแล็บ** ลองทำ `exercise.ipynb` ให้สุดก่อนเปิดไฟล์นี้

**ข้อมูล (citation):** เดียวกับ `exercise.ipynb`

In [2]:
import pandas as pd
from pathlib import Path


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)")


DATA = repo_root() / "datasets"
gap = pd.read_csv(DATA / "gapminder" / "gapminderDataFiveYear.csv")

## ส่วนที่ 1 — เลือก กรอง เรียง (Gapminder)

In [3]:
small = gap[["country", "year", "lifeExp"]]
small.head()

,country,year,lifeExp
0,Afghanistan,1952,28.801
1,Afghanistan,1957,30.332
2,Afghanistan,1962,31.997
3,Afghanistan,1967,34.020
4,Afghanistan,1972,36.088


**1.1** `df["country"]` (วงเล็บเดียว, ชื่อคอลัมน์ตัวเดียว) ได้ **Series** —
คอลัมน์เดียวทรงแท่ง ส่วน `df[["country"]]` (วงเล็บซ้อน, ชื่ออยู่ใน list)
ได้ **DataFrame** — ตารางจริงที่ยังมีโครงหนึ่งคอลัมน์ วงเล็บในคือ "ขอแบบ list"
เพราะวงเล็บนอกของ `df[...]` ถูกใช้ไปแล้ว

In [4]:
big_2002 = gap[(gap["year"] == 2002) & (gap["pop"] > 100_000_000)]
print(len(big_2002), "ประเทศ")

10 ประเทศ


In [5]:
gap_2007 = gap[gap["year"] == 2007]
low5 = gap_2007.sort_values("lifeExp", ascending=True)
low5[["country", "lifeExp"]].head()

,country,lifeExp
1463,Swaziland,39.613
1043,Mozambique,42.082
1691,Zambia,42.384
1355,Sierra Leone,42.568
887,Lesotho,42.592


**1.2** วงเล็บรอบแต่ละเงื่อนไข — เพราะ `&` ผูกแน่นกว่าตัวเปรียบเทียบ (`==`,
`>`) ถ้าไม่มีวงเล็บ Python จะตีความผิดและพัง (หรือแย่กว่านั้น: ให้ผลที่ดู
ผ่านแต่ผิด) จำสูตร: `(เงื่อนไข) & (เงื่อนไข)` เสมอ

## ส่วนที่ 2 — สรุปต่อกลุ่ม (groupby)

In [6]:
gap_2007 = gap[gap["year"] == 2007]
rich = gap_2007.groupby("continent").agg(
    mean_gdp=("gdpPercap", "mean"),
)
rich.sort_values("mean_gdp", ascending=False)

,mean_gdp
continent,
Oceania,29810.188275
Europe,25054.481636
Asia,12473.026870
Americas,11003.031625
Africa,3089.032605


**2.1** ด้วย for: หา list ชื่อทวีปก่อน (วนซ้ำค่าที่ไม่ซ้ำ) → สร้าง dict ว่าง →
วนทีละทวีป กรองแถวทวีปนั้น คำนวณค่าเฉลี่ย เก็บลง dict groupby แทนทั้งสามขั้น
(แบ่งกลุ่ม → คำนวณ → รวมกลับ) เป็นบรรทัดเดียว — และทำกับคอลัมน์/สถิติอื่น
ได้โดยไม่ต้องเขียนลูปใหม่

## ส่วนที่ 3 — รวมหลายไฟล์ (concat)

In [7]:
arabica = pd.read_csv(
    DATA / "coffee_quality" / "arabica_data_cleaned.csv")
robusta = pd.read_csv(
    DATA / "coffee_quality" / "robusta_data_cleaned.csv")

coffee = pd.concat([arabica, robusta], ignore_index=True)
print("รวมแล้ว:", coffee.shape)

รวมแล้ว: (1339, 49)


**3.1** เพราะสองไฟล์มีชุดคอลัมน์ไม่เหมือนกัน — คอลัมน์ไหนมีใน arabica แต่
ไม่มีใน robusta แถวโรบัสตาของคอลัมน์นั้นจะว่าง (NaN) ทันที เช็คได้ด้วย
`coffee.isna().mean()` (จะได้เรียนใน M3) — ที่จริงแค่สังเกต `.info()` ก็เห็น
จำนวน non-null ต่อคอลัมน์แล้ว